# Race Against the Machine (TII) + Spyx

This notebook is for the TII Race Against the Machine dataset.

Status in Tonic:
- No direct dataset class discovered in `tonic.datasets` here.

Goal:
- Provide a clean adapter path for local dataset exports.
- Benchmark multiple Spyx architectures with shared preprocessing.

In [1]:
from pathlib import Path

import haiku as hk
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

import spyx.models as fm

DATA_ROOT = Path("../data/RaceAgainstMachine_TII")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT:", DATA_ROOT.resolve())


def tonic_has_race_tii():
    import tonic.datasets as d
    return any(("race" in n.lower() and "machine" in n.lower()) or "tii" in n.lower() for n in dir(d))

print("tonic_has_race_tii =", tonic_has_race_tii())
print("expected local artifacts: events.{h5,npz}, imu.csv, pose.csv")

/Users/vincent/Work/autoresearch-mlx/.bench-venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DATA_ROOT: /Users/vincent/Work/autoresearch-mlx/spyx/research/data/RaceAgainstMachine_TII
tonic_has_race_tii = False
expected local artifacts: events.{h5,npz}, imu.csv, pose.csv


In [3]:
def synthetic_race_batch(batch=4, T=24, H=72, W=128, C=2, seed=2):
    rng = np.random.default_rng(seed)
    x = rng.poisson(0.025, size=(T, batch, H, W, C)).astype(np.float32)
    imu = rng.normal(size=(T, batch, 6)).astype(np.float32)
    traj = rng.normal(size=(T, batch, 6)).astype(np.float32)
    y = rng.normal(size=(batch, 6)).astype(np.float32)
    return jnp.asarray(x), jnp.asarray(imu), jnp.asarray(traj), jnp.asarray(y)


def race_models(input_hw=(72, 128), input_channels=2):
    imu_cfg = fm.IMUConditionedConfig(
        vision_cfg=fm.ConvConfig(input_hw=input_hw, input_channels=input_channels, channels1=24, channels2=48, output_dim=6),
        imu_dim=6,
        imu_hidden=32,
        gating="late",
    )
    rec_cfg = fm.VisualIMURecurrentConfig(
        vision_cfg=fm.ConvConfig(input_hw=input_hw, input_channels=input_channels, channels1=24, channels2=48, output_dim=32),
        imu_dim=6,
        traj_dim=6,
        hidden_dim=64,
        output_dim=6,
    )
    kf_cfg = fm.KalmanFusionConfig(latent_dim=16, output_dim=6)

    return {
        "imu_conditioned": lambda x, imu, traj: fm.IMUConditionedVisualSNN(imu_cfg)(x, imu),
        "recurrent_fusion": lambda x, imu, traj: fm.VisualIMURecurrentFusionBlock(rec_cfg)(x, imu, traj),
        "kalman_surrogate": lambda x, imu, traj: fm.KalmanStyleSpikingFusionSurrogate(kf_cfg)(
            jnp.repeat(jnp.mean(x, axis=(2, 3, 4), keepdims=False)[..., None], kf_cfg.latent_dim, axis=-1),
            jnp.repeat(jnp.mean(imu, axis=-1, keepdims=True), kf_cfg.latent_dim, axis=-1),
        ),
    }


x, imu, traj, y = synthetic_race_batch()
for name, fn in race_models().items():
    net = hk.without_apply_rng(hk.transform(fn))
    params = net.init(jax.random.PRNGKey(0), x, imu, traj)
    pred, aux = net.apply(params, x, imu, traj)
    mse = jnp.mean((pred - y) ** 2)
    print(name, pred.shape, float(mse), "spike_rate", np.asarray(aux.get("spike_rate", jnp.asarray([0.0]))))

imu_conditioned (4, 6) 1.044140338897705 spike_rate [0.00115355 0.        ]
recurrent_fusion (4, 6) 1.5018891096115112 spike_rate [0.00115355 0.        ]
kalman_surrogate (4, 6) 1.1186776161193848 spike_rate [0.]


## Optuna Sweep

This search compares Spyx fusion architectures and hidden sizes for race-style visual-inertial prediction.

In [4]:
import optuna


def race_objective(trial):
    arch = trial.suggest_categorical("arch", ["imu_conditioned", "recurrent_fusion", "kalman_surrogate"])
    channels1 = trial.suggest_categorical("channels1", [16, 24, 32])
    channels2 = trial.suggest_categorical("channels2", [32, 48, 64])
    hidden = trial.suggest_categorical("hidden", [32, 64, 96])

    x_local, imu_local, traj_local, y_local = synthetic_race_batch(batch=4, T=24, H=72, W=128, C=2, seed=trial.number)

    if arch == "imu_conditioned":
        cfg = fm.IMUConditionedConfig(
            vision_cfg=fm.ConvConfig(input_hw=(72, 128), input_channels=2, channels1=channels1, channels2=channels2, output_dim=6),
            imu_dim=6,
            imu_hidden=hidden,
            gating="late",
        )
        forward = lambda xb, ub, tb: fm.IMUConditionedVisualSNN(cfg)(xb, ub)
    elif arch == "recurrent_fusion":
        cfg = fm.VisualIMURecurrentConfig(
            vision_cfg=fm.ConvConfig(input_hw=(72, 128), input_channels=2, channels1=channels1, channels2=channels2, output_dim=hidden),
            imu_dim=6,
            traj_dim=6,
            hidden_dim=hidden,
            output_dim=6,
        )
        forward = lambda xb, ub, tb: fm.VisualIMURecurrentFusionBlock(cfg)(xb, ub, tb)
    else:
        cfg = fm.KalmanFusionConfig(latent_dim=hidden // 2 if hidden > 32 else 16, output_dim=6)
        forward = lambda xb, ub, tb: fm.KalmanStyleSpikingFusionSurrogate(cfg)(
            jnp.repeat(jnp.mean(xb, axis=(2, 3, 4), keepdims=False)[..., None], cfg.latent_dim, axis=-1),
            jnp.repeat(jnp.mean(ub, axis=-1, keepdims=True), cfg.latent_dim, axis=-1),
        )

    net = hk.without_apply_rng(hk.transform(forward))
    params = net.init(jax.random.PRNGKey(0), x_local, imu_local, traj_local)
    pred, aux = net.apply(params, x_local, imu_local, traj_local)
    mse = jnp.mean((pred - y_local) ** 2)
    spike_penalty = 0.01 * jnp.mean(jnp.asarray(aux.get("spike_rate", jnp.asarray([0.0]))))
    return float(mse + spike_penalty)


study = optuna.create_study(direction="minimize", study_name="race_tii_fusion_search")
study.optimize(race_objective, n_trials=8)
study.best_trial.params, study.best_value

[I 2026-03-24 08:03:11,057] A new study created in memory with name: race_tii_fusion_search
[I 2026-03-24 08:03:11,102] Trial 0 finished with value: 0.96941077709198 and parameters: {'arch': 'kalman_surrogate', 'channels1': 24, 'channels2': 32, 'hidden': 32}. Best is trial 0 with value: 0.96941077709198.
[I 2026-03-24 08:03:11,339] Trial 1 finished with value: 0.7513212561607361 and parameters: {'arch': 'kalman_surrogate', 'channels1': 16, 'channels2': 48, 'hidden': 64}. Best is trial 1 with value: 0.7513212561607361.
[I 2026-03-24 08:03:12,900] Trial 2 finished with value: 1.221362590789795 and parameters: {'arch': 'imu_conditioned', 'channels1': 16, 'channels2': 32, 'hidden': 64}. Best is trial 1 with value: 0.7513212561607361.
[I 2026-03-24 08:03:13,472] Trial 3 finished with value: 0.8066912293434143 and parameters: {'arch': 'imu_conditioned', 'channels1': 24, 'channels2': 64, 'hidden': 64}. Best is trial 1 with value: 0.7513212561607361.
[I 2026-03-24 08:03:14,068] Trial 4 finishe

({'arch': 'recurrent_fusion', 'channels1': 16, 'channels2': 32, 'hidden': 64},
 0.6834624409675598)

## Next Steps for Race Against the Machine (TII)

1. Wire real events/IMU/pose files into `../data/RaceAgainstMachine_TII`.
2. Add race-specific metrics (lap progression, gate consistency, drift).
3. Expand model bank with additional recurrent variants from Spyx fusion modules.